# Lab 12 — Trust but Verify
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · Week 12 lab · about 75 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/labs/lab12-trust-but-verify/Lab12_Trust_but_Verify.ipynb)

AI assistants answer engineering questions in seconds, in confident, well-formatted text. Some of those answers are wrong. When you sign a calculation, *you* are responsible for it — not the tool. Today you put a language model through nine problems where **you can compute the right answer**, score it, and practice the **5-check verification workflow** from lecture. Then you catch a bug in "AI-written" code with tests, and document your AI use.

### What you will be able to do
1. Apply the 5-check verification workflow — Restate, Units & magnitude, Independent check, Sources, Test & document — to AI answers.
2. Write short Python checks for nine common CE/ARCE calculations and score an LLM's answers.
3. Summarize which kinds of problems the model gets wrong.
4. Verify a code citation against the regulation text.
5. Write `assert` tests that catch a subtle bug in AI-written code, then fix it.
6. Write a clear AI-use acknowledgment.

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `osha/osha_1926_subpartP_excavations.txt` (29 CFR 1926 Subpart P, U.S. Government work). The language model is `Qwen/Qwen2.5-0.5B-Instruct` (about 1 GB). Model downloads happen on Google's computers, not on your laptop or phone data plan.

**AI-use label:** **AI-Assisted** — you may ask an AI tool for explanations or debugging help. Fill in the AI-use acknowledgment at the end. You are responsible for every line.

## The 5-check verification workflow
Use these five checks every time an AI tool (or a classmate, or you at 2 a.m.) gives you an engineering number.

| # | Check | Ask yourself |
|---|---|---|
| 1 | **Restate** | What is given, what is asked, in which units? Did the AI answer *that* question? |
| 2 | **Units & magnitude** | Do the units work out? Is the size reasonable (a 10 m beam with a 10⁶ kN·m moment?) |
| 3 | **Independent check** | Recompute by another route: hand calculation, Python, a worked example. |
| 4 | **Sources** | Compare with the code, standard, or textbook. Does every cited section exist and say that? |
| 5 | **Test & document** | For code: tests with known answers. Record what you checked and how you used AI. |

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

## Part 1 — Nine problems, and your own answers first (Check 3) (25 min)
Good practice: **work out the answer yourself before you read the AI's**, so its number can't anchor you. Each
problem has a **type** so we can later see *what kind* of task the model struggles with:
- **formula** — recall the right equation and plug in
- **unit conversion** — convert between unit systems
- **multi-step** — several steps that must all be right
- **arithmetic** — simple numbers, simple steps
- **code lookup** — a value from a regulation

*Units used below:* **cfs** = cubic feet per second (ft³/s), the usual U.S. unit for stormwater flow;
**kN·m** = kilonewton-meters (bending moment).

In [ ]:
problems = [
    {"id": "P1", "topic": "Beam moment", "type": "formula", "unit": "kN·m",
     "question": "A simply supported beam spans 10 m and carries a uniform load of 18 kN/m. "
                 "What is the maximum bending moment in kN·m?"},
    {"id": "P2", "topic": "psi to MPa", "type": "unit conversion", "unit": "MPa",
     "question": "Convert a concrete compressive strength of 4000 psi to MPa."},
    {"id": "P3", "topic": "Manning's equation", "type": "multi-step", "unit": "m/s",
     "question": "A rectangular concrete channel is 3.0 m wide and the water depth is 1.2 m. Manning's n = 0.013 "
                 "and the channel slope is 0.001 m/m. Use Manning's equation in SI units to find the average "
                 "flow velocity in m/s."},
    {"id": "P4", "topic": "Wall heat gain", "type": "formula", "unit": "W",
     "question": "An exterior wall in Phoenix has an area of 120 m² and a U-value of 0.35 W/(m²·K). Outdoor air "
                 "is 44 °C and indoor air is 24 °C. What is the steady-state heat gain through the wall in watts?"},
    {"id": "P5", "topic": "Water-cement ratio", "type": "unit conversion", "unit": "-",
     "question": "A concrete batch uses 12 bags of portland cement (94 lb per bag) and 60 gallons of water "
                 "(8.34 lb per gallon). What is the water-cement ratio by weight?"},
    {"id": "P6", "topic": "OSHA Type C slope", "type": "code lookup", "unit": "degrees",
     "question": "Under OSHA 29 CFR 1926 Subpart P, Appendix B, what is the maximum allowable slope for an "
                 "excavation in Type C soil that is less than 20 feet deep? Give the H:V ratio and the angle "
                 "from horizontal in degrees, and put the angle in degrees on the ANSWER line."},
    {"id": "P7", "topic": "Trip generation", "type": "arithmetic", "unit": "trips",
     "question": "A new apartment complex has 200 dwelling units. Assume a PM peak-hour trip rate of 0.55 trips "
                 "per dwelling unit, and 60% of those trips enter the site. How many entering trips are expected "
                 "in the PM peak hour?"},
    {"id": "P8", "topic": "Rational method", "type": "formula", "unit": "cfs",
     "question": "A 4.0-acre asphalt parking lot in Tucson has a runoff coefficient C = 0.90. The design rainfall "
                 "intensity is 3.0 in/hr. Use the rational method Q = CiA, where Q is in cubic feet per second "
                 "(cfs), i in in/hr, and A in acres. What is the peak runoff Q in cfs?"},
    {"id": "P9", "topic": "Factored moment", "type": "multi-step", "unit": "kN·m",
     "question": "A simply supported floor beam spans 8 m. It carries a uniform dead load of 10 kN/m (including "
                 "its self-weight) and a uniform live load of 6 kN/m. Using the strength load combination "
                 "1.2D + 1.6L (D = dead load, L = live load), what is the factored maximum bending moment in kN·m?"},
]
pd.DataFrame(problems)[["id", "topic", "type", "unit"]]

Now compute each correct answer yourself. Two are done for you as examples. Keep the units in the
comments — that is Check 2 (**Units & magnitude**) happening as you write.

In [ ]:
truth = {}   # problem id -> correct value

# P1 — simply supported beam, uniform load: M_max = q L^2 / 8   (q = distributed load, kN/m)
q, L = 18.0, 10.0                  # kN/m, m
truth["P1"] = q * L**2 / 8         # kN/m * m^2 = kN·m
print(f"P1: M_max = {truth['P1']:.1f} kN·m")

# P2 — 1 psi = 6.894757 kPa = 0.006894757 MPa   (magnitude check: 1 MPa ≈ 145 psi)
truth["P2"] = 4000 * 0.006894757   # MPa
print(f"P2: 4000 psi = {truth['P2']:.2f} MPa")

**P3 — Manning's equation (SI units):** V = (1/n) · R^(2/3) · S^(1/2), where R is the **hydraulic
radius** = flow area A ÷ wetted perimeter P. For a rectangular channel of width b and depth y: A = b·y and
P = b + 2y (the bottom plus two sides — the free surface is not "wetted").

> 💡 **Hint:** `R ** (2 / 3)` is R to the power 2/3; `S ** 0.5` is the square root.

In [ ]:
# TODO: compute A, P, R, and V for P3 (b = 3.0 m, y = 1.2 m, n = 0.013, S = 0.001).
b, y, n, S = 3.0, 1.2, 0.013, 0.001
A = None     # m²
P = None     # m
R = None     # m
truth["P3"] = None    # m/s
print(f"P3: R = {R:.3f} m, V = {truth['P3']:.3f} m/s")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert abs(truth['P3'] - 1.856) < 0.005, 'Expected about 1.856 m/s.'
print('Correct!')

> 💡 **Hint:** Multiply for P4. For P5, convert both to pounds first, then divide water by cement.

In [ ]:
# TODO: P4 heat gain Q = U * A * ΔT, and P5 water-cement ratio (both in pounds).
U, A_wall, dT = 0.35, 120.0, 44 - 24     # W/(m²·K), m², K (a difference of 1 °C equals 1 K)
truth["P4"] = None    # W

water_lb = None       # 60 gal × 8.34 lb/gal
cement_lb = None      # 12 bags × 94 lb/bag
truth["P5"] = None    # dimensionless
print(f"P4: Q = {truth['P4']:.0f} W    P5: w/c = {truth['P5']:.3f}")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert abs(truth['P4'] - 840) < 1 and abs(truth['P5'] - 0.4436) < 0.001
print('Correct!')

**P6 — a code lookup.** You can't compute a regulation — you look it up in the **source** (Check 4).
The next cell searches the OSHA text for Table B-1.

In [ ]:
osha = open(get_data("osha/osha_1926_subpartP_excavations.txt"), encoding="utf-8").read()
for line in osha.splitlines():
    if line.startswith(("TABLE B-1", "Stable Rock |", "Type A (", "Type B |", "Type C |")):
        print(line[:120])

> 💡 **Hint:** `math.atan(x)` returns radians; wrap it in `math.degrees(...)`.

In [ ]:
# TODO: Table B-1 says Type C = 1 1/2 H : 1 V. Compute that angle from horizontal in degrees.
# The slope rises 1 (V) over a run of 1.5 (H), so tan(angle) = V / H.
angle = None
truth["P6"] = 34     # OSHA's rounded value, from the source
print(f"P6: atan(1/1.5) = {angle:.1f}°; OSHA lists {truth['P6']}°")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert abs(angle - 33.69) < 0.05
print('Correct!')

**P8 — rational method, unit convention.** In U.S. units the rational method is written
**Q (cfs) ≈ C · i (in/hr) · A (acres)**. It works because 1 acre·in/hr = 1.008 ft³/s, so the conversion factor is
almost exactly 1. (In SI it becomes Q (m³/s) = C · i (mm/h) · A (ha) / 360.)

**P9 — a real multi-step check.** First the factored load q_u = 1.2 × dead + 1.6 × live (kN/m), then
M_u = q_u · L² / 8.

> 💡 **Hint:** Just multiply the given numbers. 60% = 0.60. For P9, compute q_u first, then use the beam formula.

In [ ]:
# TODO: P7 entering trips, P8 rational-method runoff, and P9 factored moment.
truth["P7"] = None    # trips = units × rate × share entering
truth["P8"] = None    # cfs  = C × i × A
q_u = None            # kN/m, factored load = 1.2 × 10 + 1.6 × 6
truth["P9"] = None    # kN·m, q_u × span² / 8 with span = 8 m
print(f"P7: {truth['P7']:.0f} entering trips    P8: Q = {truth['P8']:.1f} cfs    P9: M_u = {truth['P9']:.1f} kN·m")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert abs(truth['P7'] - 66) < 0.01 and abs(truth['P8'] - 10.8) < 0.01 and abs(truth['P9'] - 172.8) < 0.01
print('Correct!')

## Part 2 — Ask the AI and score it (20 min)
Load the small language model from Lab 01 (about 1 GB; model downloads happen on Google's computers, not on your
laptop or phone data plan).

In [ ]:
from transformers import pipeline
chat = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")

def ask(question, max_new_tokens=500):
    """Send one question to the small language model and return its answer as text."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant for civil and architectural engineering students."},
        {"role": "user", "content": question},
    ]
    out = chat(messages, max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

FORMAT = (" Show the formula and your steps briefly. End with one final line in exactly this form: "
          "ANSWER: <number> <unit>")
ai_answers = {}    # problem id -> the model's answer text (kept, so re-running the next cell doesn't ask again)

The next cell asks all nine questions. On Colab's free CPU it takes **about 10–20 minutes** (2–4 minutes
on a T4 GPU), and Colab can't run other cells meanwhile — use the time to read Parts 3 and 4. Answers are saved in
`ai_answers`, so if you re-run the cell it only asks the questions that are still missing.

In [ ]:
for p in problems:
    if p["id"] not in ai_answers:                      # ask only once
        ai_answers[p["id"]] = ask(p["question"] + FORMAT)
    p["ai_text"] = ai_answers[p["id"]]
    print(f"=== {p['id']} {p['topic']} ===")
    print(p["ai_text"], "\n")

Python reads the number on the `ANSWER:` line with a **regular expression** (a text pattern:
`\d` means "a digit"). **Heads-up:** this small model usually ignores the `ANSWER:` instruction, so
`extract_answer` often falls back to *the last number in the text* — which may not be the model's answer at all.
That is Check 5 applied to your own checking code: look at every reading yourself.

In [ ]:
def extract_answer(text):
    """Return the number after 'ANSWER:' (or the last number in the text), or None if there is no number."""
    number = r"[-+]?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?"
    m = re.search(r"ANSWER\W*?\s*(" + number + ")", text, flags=re.IGNORECASE)
    found = [m.group(1)] if m else re.findall(number, text)
    if not found:
        return None
    return float(found[-1].replace(",", ""))

for p in problems:
    p["ai_value"] = extract_answer(p["ai_text"])
    print(p["id"], p["topic"], "->", p["ai_value"], p["unit"])

### Score the AI
We call an AI answer **correct** if its number is within **2%** of your correct value (rounding is fine; a wrong
formula or unit is not).

In [ ]:
def is_correct(ai_value, true_value, tol=0.02):
    return ai_value is not None and abs(ai_value - true_value) <= tol * abs(true_value)

rows = []
for p in problems:
    rows.append({"id": p["id"], "topic": p["topic"], "type": p["type"], "unit": p["unit"],
                 "AI answer": "none" if p["ai_value"] is None else f"{p['ai_value']:,.4g}",
                 "correct value": f"{truth[p['id']]:,.4g}",
                 "AI correct?": is_correct(p["ai_value"], truth[p["id"]])})
score = pd.DataFrame(rows)
print(f"The model got {score['AI correct?'].sum()} of {len(score)} right.")
score

In [ ]:
by_type = score.groupby("type")["AI correct?"].agg(correct="sum", total="count")
by_type["wrong"] = by_type["total"] - by_type["correct"]
print(by_type)

ax = by_type[["correct", "wrong"]].plot.barh(stacked=True, color=["#70B865", "#AB0520"], figsize=(6, 3))
ax.set_xlabel("Number of problems")
ax.set_ylabel("Problem type")
ax.set_title(f"Qwen2.5-0.5B on {len(problems)} engineering problems")
ax.set_xticks(range(0, int(by_type["total"].max()) + 1))
plt.tight_layout(); plt.show()

**✍️ Question:** Look at the table. Did `extract_answer` read the number you would have read for every problem? If not, which one, and does it change the score? (Check 5 applies to your *own* checking code too.)

*Your answer:* 

**✍️ Question:** Pick one problem the model got wrong. Which of the 5 checks would catch the error fastest, and what exactly went wrong (formula, arithmetic, units, a made-up value, or answering a different question)?

*Your answer:* 

**✍️ Question:** For a problem the model got right: does that mean you can skip checking next time? Explain in 1–2 sentences.

*Your answer:* 

### Optional: a much bigger model (Colab only)
A 0.5-billion-parameter model is an easy target. If Colab's built-in Gemini is available to you, the next cell
asks it the same nine questions and scores them the same way. (If it prints a message instead, skip it.) Even if a
big model gets all nine right, keep checking: its mistakes are rarer and **subtler** — exactly the kind you will
hunt for in HW5 Part A.

In [ ]:
try:
    from google.colab import ai
    rows_big = []
    for p in problems:
        text = ai.generate_text(p["question"] + FORMAT)
        value = extract_answer(text)
        rows_big.append({"id": p["id"], "topic": p["topic"], "Gemini answer": value,
                         "correct value": round(truth[p["id"]], 3), "Gemini correct?": is_correct(value, truth[p["id"]])})
    display(pd.DataFrame(rows_big))
except Exception as err:
    print("Colab's built-in model is not available here — skip this optional step.", type(err).__name__)

## Part 3 — Check 4: Sources (10 min)
AI tools often **cite** sections of codes. A citation can look perfect and still not exist, or exist but say
something else. Ask the model for a citation, then search the actual regulation text for it.

In [ ]:
cite_answer = ask("Which section of OSHA 29 CFR 1926 Subpart P gives the maximum allowable slopes for "
                  "sloping systems in Type C soil? Cite the section number and quote what it says.")
print(cite_answer)

In [ ]:
# Find every section-like number in the answer (e.g., "1926.652(b)(1)" or "4.3.5") and search the text for it.
cited = sorted(set(re.findall(r"\b(?:1926\.\d+(?:\([a-z0-9]+\))*|\d+\.\d+\.\d+(?:\.\d+)*)", cite_answer)))
print("Section numbers cited in the AI answer:", cited)
for c in cited:
    base = c.split("(")[0]              # "1926.652(b)(1)" -> "1926.652" (paragraph letters are checked by reading)
    print(f"  {c:20s} section {base} appears in the OSHA Subpart P text? {base in osha}")
print("Real section numbers in the text look like:", sorted(set(re.findall(r"§ 1926\.\d+", osha))))

Our search only checks that the **section number exists** — the cheapest check. It cannot tell whether
the quote is accurate. For that, read the source. The next cell prints what § 1926.652(b)(1) and Table B-1 really
say.

In [ ]:
for line in osha.splitlines():
    if line.startswith("(1) Option (1)—Allowable configurations") or line.startswith("Type C |"):
        print(line, "\n")

**✍️ Question:** Compare the AI's citation and quote with the real text. Is the section number real? Is the quote accurate, partly accurate, or invented?

*Your answer:* 

## Part 4 — Check 5: Test AI-written code (15 min)
A classmate asked an AI assistant for a Manning's-equation function and pasted this into the team notebook. It
has a docstring, units, and clean style. **Read it before you run it.**

In [ ]:
# "AI-written" code — pasted as-is from a chat assistant
def manning_velocity_ai(n, R, S):
    """Average flow velocity (m/s) from Manning's equation, SI units.

    n: Manning roughness coefficient (-), R: hydraulic radius (m), S: channel slope (m/m).
    """
    return (1.0 / n) * R ** (3 / 2) * S ** 0.5

print(f"{manning_velocity_ai(0.013, 1.0, 0.001):.4f} m/s")

Your classmate "tested" it with R = 1 m and got the textbook value 2.4325 m/s, so it must be right…
right? Look at the exponent of R again. **1 raised to any power is 1**, so a test with R = 1 m can never catch a
wrong exponent. Good tests are chosen so that a *wrong* function would *fail* them.

An **`assert`** statement checks that something is true and stops with an error message if it isn't:
`assert abs(value - expected) < tolerance, "message"`.

> 💡 **Hint:** Use the same pattern as Test 1: `assert abs(f(n, R, S) - expected) < 0.001, "message"`. For Test 3 compute `f(0.013, 2.0, 0.001) / f(0.013, 1.0, 0.001)` and compare it with `2 ** (2 / 3)`.

In [ ]:
# TODO: add two tests to test_manning(f), where f is the function under test.
def test_manning(f):
    # Test 1 (given): R = 1 m. Passes even with a wrong exponent!
    assert abs(f(0.013, 1.0, 0.001) - 2.4325) < 0.001, "R = 1 m case failed"

    # TODO Test 2: the Problem 3 channel (n = 0.013, R = 3.6/5.4 m, S = 0.001) should give truth["P3"] (about 1.856 m/s)

    # TODO Test 3: doubling R should multiply V by 2 ** (2/3) (about 1.587)

    print("All tests passed.")

try:
    test_manning(manning_velocity_ai)
except AssertionError as err:
    print("BUG CAUGHT:", err)

> 💡 **Hint:** Manning's equation uses R to the power **2/3**, not 3/2.

In [ ]:
# TODO: fix the function, then run the same tests on it.
def manning_velocity(n, R, S):
    """Average flow velocity (m/s) from Manning's equation, SI units."""
    return (1.0 / n) * R ** (3 / 2) * S ** 0.5    # <- fix this line

test_manning(manning_velocity)

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert abs(manning_velocity(0.013, 3.6 / 5.4, 0.001) - 1.856) < 0.005
print('Fixed!')

**✍️ Question:** The buggy function gave 1.856 m/s → what value instead for the Problem 3 channel? Would a units & magnitude check alone have caught it? Why are tests better than one spot check?

*Your answer:* 

### Optional: ask the model to write the function
"Read before you run": the next cell only **prints** the model's code — it does not execute it.

In [ ]:
code_answer = ask("Write a short Python function manning_velocity(n, R, S) that returns the flow velocity "
                  "in m/s from Manning's equation in SI units. Only give the code.", max_new_tokens=200)
print(code_answer)

**✍️ Question:** Read the model's code. Is the exponent on R correct? Does it use the SI form (1/n) or the U.S. form (1.49/n)? Would your tests catch any problem you see?

*Your answer:* 

## Part 5 — Check 5: Document (5 min)
The last check is writing down what you did. The course uses one **AI-use acknowledgment** template (see the
[AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md)):

> I acknowledge the use of **[tool and version]** to **[specific use]**. Prompts I used included **[paste or
> summarize]**. I checked the output by **[how I verified it]**. I am responsible for this work.

A filled-in example, based on today's Part 4:

> *I acknowledge the use of **[chat assistant, version]** to **draft a Python function `manning_velocity` for
> Manning's equation in SI units**. Prompts I used included **"Write a Python function for Manning's equation in SI
> units."** I checked the output by **comparing the formula with my course notes and writing three `assert`
> tests; a test with R = 1 m passed even though the exponent on R was wrong (3/2 instead of 2/3), so I added a test
> with a hand-calculated channel (R = 0.667 m), which caught the bug, and I fixed the exponent**. I am responsible
> for this work.*

**✍️ Question:** Write a verification log for today: (1) which AI tool you used and for what, (2) how many of the 9 answers were correct and which types failed, (3) which checks you used, and (4) one rule you will follow when using AI for engineering calculations.

*Your answer:* 

---
## 🎟️ Exit ticket
Three short questions you answer **alone, without AI (AI-Free)**, during the last 10 minutes of lab — even if you have not finished every part above. Use your own words, 1–3 sentences each. (If you must leave early, answer them on your own before you submit.)

**1.** Name the 5 checks of the verification workflow, in order.

*Your answer:* 

**2.** Why did the test with R = 1 m fail to catch the bug in `manning_velocity_ai`?

*Your answer:* 

**3.** An AI answer cites "OSHA 1926.652(b)(5)". What exactly would you do to verify it?

*Your answer:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
**Due:** Sunday 11:59 pm (Arizona time) of this week, on Brightspace. Late work: see the syllabus late-work table (two grace days per semester, then −10% per day, up to 3 days).

1. **Runtime → Restart session and run all** — make sure everything runs without errors (cells that load a model take a few extra minutes).
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.